In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

In [0]:
# Checkpoint directory for structured streaming

dbutils.fs.mkdirs("/Volumes/lakehouse/01_raw/raw/_checkpoint")

In [0]:
source_path = '/Volumes/lakehouse/01_raw/raw/'
checkpoint_path = '/Volumes/lakehouse/01_raw/raw/_checkpoint'

In [0]:
# Read the json data without schema defined- > will fail when trying to display
# df = (spark.readStream
#       .format("json")
#       .load(source_path+"/*.json"))

In [0]:
# Schema from json file is complex as it has json, with nested dictionary and lists
# to fix issue from previous code block

df = spark.read.option('inferSchema', True).json(path='/Volumes/lakehouse/01_raw/raw/earthquake_data_2026-03-26-19-14-37.json')

# Now we have shema defined


In [0]:
df.schema
# Copy the output of this command and paste into next cell then tell AI Assist to 'format this schema'

In [0]:
source_schema = StructType([
    StructField('bbox', ArrayType(DoubleType(), True), True),
    StructField('features', ArrayType(
        StructType([
            StructField('geometry', StructType([
                StructField('coordinates', ArrayType(DoubleType(), True), True),
                StructField('type', StringType(), True)
            ]), True),
            StructField('id', StringType(), True),
            StructField('properties', StructType([
                StructField('alert', StringType(), True),
                StructField('cdi', DoubleType(), True),
                StructField('code', StringType(), True),
                StructField('detail', StringType(), True),
                StructField('dmin', DoubleType(), True),
                StructField('felt', LongType(), True),
                StructField('gap', LongType(), True),
                StructField('ids', StringType(), True),
                StructField('mag', DoubleType(), True),
                StructField('magType', StringType(), True),
                StructField('mmi', DoubleType(), True),
                StructField('net', StringType(), True),
                StructField('nst', LongType(), True),
                StructField('place', StringType(), True),
                StructField('rms', DoubleType(), True),
                StructField('sig', LongType(), True),
                StructField('sources', StringType(), True),
                StructField('status', StringType(), True),
                StructField('time', LongType(), True),
                StructField('title', StringType(), True),
                StructField('tsunami', LongType(), True),
                StructField('type', StringType(), True),
                StructField('types', StringType(), True),
                StructField('tz', StringType(), True),
                StructField('updated', LongType(), True),
                StructField('url', StringType(), True)
            ]), True),
            StructField('type', StringType(), True)
        ]), True
    ), True),
    StructField('metadata', StructType([
        StructField('api', StringType(), True),
        StructField('count', LongType(), True),
        StructField('generated', LongType(), True),
        StructField('status', LongType(), True),
        StructField('title', StringType(), True),
        StructField('url', StringType(), True)
    ]), True),
    StructField('type', StringType(), True)
])

In [0]:
# Now that we have our schema defined, we can read the data
df = (spark.readStream
      .format("json")
      .schema(source_schema)
      .load(source_path+"/*.json"))

In [0]:
# Now we can write the data to bronze schema, create an earthquake table if not there or write to it if there already and checkpoint is updated

# If checkpoint folder is removed from raw volume then next time the stream is written, it will process all the json files in the raw volume instead of the latest. This is because the stream will not know which files it has already processed.

query = (df.writeStream
         .option('checkpointLocation', checkpoint_path)
         .trigger(availableNow=True) # Process all new files then stop (Batch style)
         .toTable('lakehouse.02_bronze.earthquake'))